# Notebook zur erstellung des Datasets

In [126]:
import os
import random
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter

### Parameter

In [127]:
DATA_DIR = "../sensor_and_video_data"
LABELED_DIR = "../Labeled"
OUTPUT_DIR = "../datasets_output"
SEED = 42
# Files expeted in a run folder for it to be considered complete
EXPECTED_FILES = ["selected_peaks", "synch_data", "Timestamps"]

CONVERSION_FACTOR = 2.4  # To convert from video frames to sensor samples


# Split ratios
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Classe labels
CLASS_LABELS = {
        "Jumping" : 0,
        "Back Wheel block" : 1,
        "Right" : 2,
        "Left" : 3,
        "Sonstige" : 4,
        "Nothing" : 5
}

# Dataset creation parameters
WINDOW_SIZE = 50  # Number of samples in each window
STEP_SIZE = 10    # Step size for sliding window


### Helper functions

In [128]:
def check_run_complete(folder, keywords):
    files = os.listdir(os.path.join(DATA_DIR, folder))

    for word in keywords:
        if not any(word in file for file in files):
            return False
    return True

### Runweise Aufteilung in Train/Validation/Test + auschließen von unvollständigen Runs

In [129]:

# Collect all run directories
runs = [
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
]

# Exclude incomplete runs
for run in runs[:]:
    if not check_run_complete(run, EXPECTED_FILES):
        runs.remove(run)
        print(f"Removed incomplete run: {run}")

print(f"Found {len(runs)} runs.")

# Shuffle runs for randomness
random.seed(SEED)
random.shuffle(runs)

# Split into train, val, test
num_runs = len(runs)
train_split = int(TRAIN_SPLIT * num_runs)
val_split = int((TRAIN_SPLIT + VAL_SPLIT) * num_runs)
train_runs = runs[:train_split]
val_runs = runs[train_split:val_split]
test_runs = runs[val_split:]

print(train_runs)
print(val_runs)
print(test_runs)

Found 16 runs.
['0721_0853_noheadsensor', '0724_0723', '0720_0828', '0720_0858', '0802_0800', '0725_0709', '0727_0812', '0721_0928_noheadsensor', '0713_0903', '0713_0932', '0801_0758']
['0803_0734', '0720_0748']
['0727_0735', '0713_0833', '0717_0717']


### Funktion zur Erstellen des Label Vektors

Encoding: siehe CLASS_LABELS oben.
Bezieht sich auf Video Frames und wird nur Umgerechnet !!!

In [130]:
def create_label_vector(run_file, len_of_run):
    file_path = os.path.join(LABELED_DIR, run_file)
    if os.path.isfile(file_path):
        print(f"Label file exists for {run_file}")
        label_vector = ["Nothing"] * len_of_run

        with open(file_path, 'r') as f:
            data = json.load(f)

        raw = data["button_presses"]
        # *2.4 to convert from video frames to sensor samples
        sequence_width_half = int(int(data["sequence_width"])*CONVERSION_FACTOR) // 2

        button_presses = []
        for entry in raw.split(";"):
            entry = entry.strip()
            if not entry:
                continue
            label, idx = entry.split(":")
            button_presses.append((int(float(idx.strip()) * CONVERSION_FACTOR), label.strip())) # convert from video frames to sensor samples

        # Insert labels at the correct positions in the label vector
        for idx, label in button_presses:
            if 0 <= idx < len_of_run:
                label_vector[idx-sequence_width_half : idx + sequence_width_half + 1] = [label] * (2 * sequence_width_half + 1)

        # Convert string labels to int labels
        label_vector = [CLASS_LABELS[label] for label in label_vector]
        return label_vector
        
    else:
        print(f"Label file missing for {run_file}")


create_label_vector("0717_0717_sequences.json", 50000)
    

Label file exists for 0717_0717_sequences.json


[5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,


### Funktion zur Synchronisation von Video- und Sensordaten

Synchronisation des Videos und der Sensordaten durch die Dateie synch_data.json und selected_peaks.json. In selected_peaks.json sind die PacketCounter werte zu finden and denen die verschiedenen Sensoren geklopft wurden. In Timestamps.json sind die dazugehörigen Video Frames. Start des Sliding Windows nach klopfen des letzten Sensors

In [131]:
def synch_data(run_file):
    '''Function which returns the start values for the video file/label vector and the sensor data file.
    when startet from these values both data streams are synchronized. Values are returned in number of samples, NOT 
    video frames.'''

    file_path_synch = os.path.join(DATA_DIR, run_file, "synch_data.json")
    file_path_selected_peaks = os.path.join(DATA_DIR, run_file, "selected_peaks.json")

    # Load synch data
    if os.path.isfile(file_path_synch) and os.path.isfile(file_path_selected_peaks):
        print(f"Synch file and selected peaks file exists for {run_file}")

        with open(file_path_synch, 'r') as f:
            data_synch = json.load(f)

        with open(file_path_selected_peaks, 'r') as f:
            data_selected_peaks = json.load(f)

        # Extract timestamps and convert to sensor samples if needed
        try:
            sensor_timestamps_synch_data =  [int(data_synch["Head Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR),
                                            int(data_synch["Wrist Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR),
                                            int(data_synch["Seat Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR)]
            sensor_timestamps_selected_peaks = [data_selected_peaks["XSens"]["Head"],
                                                data_selected_peaks["XSens"]["Wrist"],
                                                data_selected_peaks["XSens"]["Seat"]]
        except KeyError as e:
            print(f"KeyError: {e} in synch or selected peaks file for {run_file}.")
            return None, None


        # Determine start values -> the last sensor that was tapped determines the start value for both streams
        start_value_video = max(sensor_timestamps_synch_data)
        start_value_sensor = max(sensor_timestamps_selected_peaks)
        print(f"Start value video/label vector: {start_value_video}, Start value sensor: {start_value_sensor}")

        return start_value_video, start_value_sensor

    else:
        print(f"Synch or selected peaks file missing for {run_file}")
        return None, None

    
synch_data("0717_0717")   


Synch file and selected peaks file exists for 0717_0717
Start value video/label vector: 2544, Start value sensor: 1344


(2544, 1344)

### Funktion um Daten zu laden und in einem Dataframe zusammenzufassen

In [132]:
def load_and_merge_data(run_file,
                        sensor_prefixes=("Wrist", "Head", "Seat"),
                        time_col="PacketCounter",
                        allow_missing=False,
                        fill_missing_with_zeros=False,
                        channels_per_sensor=9):
    
    '''Load and merge sensor data from multiple CSV files in a run folder. If one of the sensor files is missing,
    the user can decide to either raise an error, or fill the missing data with zeros.'''
    
    # Get all csv files in the run folder
    run_path = os.path.join(DATA_DIR, run_file)
    run_path = Path(run_path)
    files = list(run_path.glob("*.csv"))

    data_frames = {}
    base_time = None
    # Columns to be used because some csv files have a , at the end which causes issues
    # also to exclude SampleTimeFine because it is not needed
    cols = ["PacketCounter","Euler_X","Euler_Y","Euler_Z",
        "Acc_X","Acc_Y","Acc_Z","Gyr_X","Gyr_Y","Gyr_Z"]

    for prefix in sensor_prefixes:
        matches = [f for f in files if f.name.startswith(prefix)]
         
        # Handling when files are missing.
        if len(matches) == 0:
            if not allow_missing:
                print(f"No file found for sensor '{prefix}' in run '{run_file}'. Returning empty DataFrame.")
                return pd.DataFrame()  # Return empty DataFrame
            
            if fill_missing_with_zeros:
                print(f"Filling missing sensor data for '{prefix}' with zeros.")
                if base_time is None:
                    raise ValueError("Base time is not set. Cannot create zero-filled DataFrame.")
                dummy = pd.DataFrame(
                    np.zeros((len(base_time), channels_per_sensor)),
                    columns=[f"{prefix.lower()}_ch{i}" for i in range(channels_per_sensor)]
                )
                dummy[time_col] = base_time
                data_frames[prefix.lower()] = dummy
                continue
        
        # Determine start line of header
        with open(matches[0], "r") as f:
            for i, line in enumerate(f):
                if line.startswith("PacketCounter"):
                    header_line = i
                    break
        # read csv file, skip unneeded rows
        df = pd.read_csv(matches[0], skiprows=header_line, usecols=cols)
        df = df.add_prefix(f"{prefix.lower()}_")
        df = df.rename(columns={f"{prefix.lower()}_{time_col}": time_col})
        data_frames[prefix.lower()] = df

        # set the base time from the first available sensor
        if base_time is None:
            base_time = df[time_col]

    # Merge all data frames on the time column
    merged = data_frames[sensor_prefixes[0].lower()]
    for prefix in sensor_prefixes[1:]:
        if prefix.lower() in data_frames:
            merged = pd.merge(merged, data_frames[prefix.lower()], on=time_col)

    merged.set_index(time_col, inplace=True)
    merged.head()

    # remove nan and infinite values
    merged = merged.fillna(0)
    merged = merged.replace([np.inf, -np.inf], [1e10, -1e10])
    # smooth out the sensor data to eliminate outliers. Need to check the impact of this later
    if not merged.empty and len(merged) > 50:
        df_smoothed = pd.DataFrame(
            savgol_filter(merged.values, window_length=5, polyorder=2, axis=0),
            columns=merged.columns,
            index=merged.index
       )
        return df_smoothed

    return merged


load_and_merge_data("0717_0717")

   

,wrist_Euler_X,wrist_Euler_Y,wrist_Euler_Z,wrist_Acc_X,wrist_Acc_Y,wrist_Acc_Z,wrist_Gyr_X,wrist_Gyr_Y,wrist_Gyr_Z,head_Euler_X,...,head_Gyr_Z,seat_Euler_X,seat_Euler_Y,seat_Euler_Z,seat_Acc_X,seat_Acc_Y,seat_Acc_Z,seat_Gyr_X,seat_Gyr_Y,seat_Gyr_Z
PacketCounter,,,,,,,,,,,,,,,,,,,,,
0,-95.131921,-12.473101,-0.617245,0.225951,-1.068407,-0.106980,-0.387022,0.084789,0.262242,-3.273018,...,0.009626,6.616719,-60.784750,-148.589862,0.996008,0.065041,0.553973,0.140980,0.006587,0.032916
1,-95.186462,-12.427215,-0.602498,1.465929,-7.073697,-0.788212,-1.909607,0.727580,1.832761,-3.309911,...,-0.462957,6.637931,-60.791732,-148.606781,6.448652,0.435703,3.568829,0.830442,-0.048727,0.309242
2,-95.279995,-12.360129,-0.593770,2.153308,-10.356822,-1.171539,-2.391482,1.220615,2.647074,-3.316340,...,-0.844199,6.665147,-60.800079,-148.627225,9.423185,0.631851,5.217058,1.200253,-0.065648,0.436859
3,-95.432498,-12.259225,-0.584910,2.009253,-9.577771,-1.125987,-1.326595,1.475753,2.356934,-3.297432,...,-1.145785,6.704527,-60.811931,-148.658419,8.675348,0.571346,4.808926,1.086625,-0.037843,0.366359
4,-95.521277,-12.193699,-0.598657,2.037649,-9.558495,-1.118018,-0.515224,1.819514,2.254369,-3.250640,...,-1.344922,6.729290,-60.817538,-148.670076,8.683658,0.550897,4.828802,1.056512,-0.012974,0.278713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56008,-97.297585,-14.534778,124.521490,1.934520,-8.998656,-1.140016,-31.740660,-16.719739,8.160757,-5.265621,...,8.114705,27.073062,-56.970888,64.785094,8.411005,2.080882,4.703327,1.571003,0.060978,-0.977498
56009,-97.647084,-14.458845,124.659424,2.003066,-9.252435,-1.405830,-37.540513,-16.611910,7.665341,-5.257936,...,8.640157,27.095528,-56.963765,64.763346,8.406651,2.017114,4.704630,1.727853,0.118470,-1.207749
56010,-98.028561,-14.388696,124.791306,2.099412,-9.498283,-1.760738,-41.623889,-15.892180,7.001415,-5.250755,...,9.027491,27.122030,-56.955404,64.739416,8.414629,1.999226,4.733121,1.872385,0.143420,-1.479828


### Funktion zur erstellung des Datensatzes + Label Datensatz

X und y werden zu Numpy array konvertiert um es einheitlich zu halten.

In [133]:
def create_dataset(run_file, window_size=50, step_size=10):

    '''Create dataset samples using a sliding window approach from the sensor data and label vector of a given run.
    Returns two numpy ndarrays: X (sensor data samples) and y (corresponding labels).'''
    # Load and merge sensor data
    sensor_data = load_and_merge_data(run_file)
    if sensor_data.empty:
        print(f"Skipping run {run_file} due to missing sensor data.")
        return None, None
    

    len_of_run = len(sensor_data)

    # Create label vector
    label_vector = create_label_vector(f"{run_file}_sequences.json", len_of_run)
    if label_vector is None:
        print(f"Skipping run {run_file} due to missing label data.")
        return None, None

    # get synchronization points
    start_video, start_sensor = synch_data(run_file)
    if start_video is None or start_sensor is None:
        print(f"Skipping run {run_file} due to missing synchronization data.")
        return None, None

    # Align sensor data and label vector based on starting points of the synch_data function
    end_sensor = start_sensor + (len_of_run - start_video)
    sensor_data_synced = sensor_data.iloc[start_sensor:end_sensor].reset_index(drop=True)
    label_vector_synced = label_vector[start_video:start_video + len(sensor_data_synced)]

    X = []
    y = []

    # Sliding window to create samples
    for start in range(0, len(sensor_data_synced) - window_size + 1, step_size):
        end = start + window_size
        mid_index = start + window_size // 2

        X.append(sensor_data_synced.iloc[start:end].values)
        y.append(label_vector_synced[mid_index])  # Label at the middle of the window

    # convert to numpy arrays
    X = np.array(X)
    y = np.array(y)

    print(f"Created {len(X)} samples from run {run_file}")
    print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")
    return X, y

create_dataset("0717_0717", window_size=WINDOW_SIZE, step_size=STEP_SIZE)


Label file exists for 0717_0717_sequences.json
Synch file and selected peaks file exists for 0717_0717
Start value video/label vector: 2544, Start value sensor: 1344
Created 5342 samples from run 0717_0717
Shape of X: (5342, 50, 27), Shape of y: (5342,)


(array([[[-1.23267328e+01,  6.23051305e+00, -5.09535538e+01, ...,
          -3.51870053e-02, -1.45747548e-01,  1.71848309e+00],
         [-1.22594280e+01,  6.34737611e+00, -5.08941984e+01, ...,
          -2.35252114e-02, -1.10778263e-01,  1.64871282e+00],
         [-1.22104271e+01,  6.47054544e+00, -5.08342722e+01, ...,
           6.70789203e-02, -1.14478430e-01,  1.60074325e+00],
         ...,
         [-6.76206601e+00,  2.22730305e+01, -4.31280197e+01, ...,
           4.38853784e+00,  2.59802983e-01, -5.62697570e+00],
         [-6.56667799e+00,  2.27248650e+01, -4.28475061e+01, ...,
           4.32681916e+00,  2.52707852e-01, -5.58511015e+00],
         [-6.36347667e+00,  2.31676657e+01, -4.25619156e+01, ...,
           4.22377951e+00,  2.74888314e-01, -5.53980973e+00]],
 
        [[-1.15442276e+01,  7.64891089e+00, -5.01114930e+01, ...,
           6.30828813e-01, -6.99213701e-02,  5.63319227e-01],
         [-1.14419174e+01,  7.82961277e+00, -4.99898197e+01, ...,
           7.11179919

### Scaling

In [ ]:
def scale_features(X, scaler=None):
    n_X , window_size, n_features = X.shape
    # flatten data into 2d shape for the scaler to be able to work
    X_2d = X.reshape(n_X*window_size, n_features)

    #X_2d = np.nan_to_num(X_2d, nan=0.0, posinf=1e10, neginf=-1e10)

    # check if there is a scaler -> important for test and val runs because they need to use the same scaler as train data
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(X_2d)
        print(f"Scaler successfully fitted")
        for i, (mu, sigma) in enumerate(zip(scaler.mean_, scaler.scale_)):
            print(f"Feature {i:02d}: mean = {mu:.4f}, std = {sigma:.4f}")

    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(n_X, window_size, n_features), scaler
    


### Kombination von allen Runs

In [135]:
print("################## Training Set ##################")
X_train = []
y_train = []
i = 0
for run in train_runs:
    X_run, y_run = create_dataset(run, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
    if X_run is not None and y_run is not None:
        if len(X_train) == 0:
            X_train = X_run
            y_train = y_run
            i += 1
            print("-------------------------------------------")
            continue
        X_train = np.vstack((X_train, X_run))
        y_train = np.hstack((y_train, y_run))
        i += 1
    print("-------------------------------------------")
X_train_scaled, scaler = scale_features(X_train)
print(f"{i} runs out of {len(train_runs)} processed for training set.")
i = 0

print("\n\n################## Validation Set ##################")

X_val = []
y_val = []
for run in val_runs:
    X_run, y_run = create_dataset(run, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
    if X_run is not None and y_run is not None:
        if len(X_val) == 0:
            X_val = X_run
            y_val = y_run
            i += 1
            print("-------------------------------------------")
            continue
        X_val = np.vstack((X_val, X_run))
        y_val = np.hstack((y_val, y_run))
        i += 1
    print("-------------------------------------------")
    X_val_scaled, _ = scale_features(X_val, scaler)
print(f"{i} runs out of {len(val_runs)} processed for validation set.")
i = 0

print("\n\n################## Test Set ##################")

X_test = []
y_test = []
for run in test_runs:
    X_run, y_run = create_dataset(run, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
    if X_run is not None and y_run is not None:
        if len(X_test) == 0:
            X_test = X_run
            y_test = y_run
            i += 1
            print("-------------------------------------------")
            continue
        X_test = np.vstack((X_test, X_run))
        y_test = np.hstack((y_test, y_run))
        i += 1
    print("-------------------------------------------")
X_test_scaled, _ = scale_features(X_test, scaler)
print(f"{i} runs out of {len(test_runs)} processed for test set.")
i = 0

print("\n\nSaving datasets to datasets_output...")
np.savez(OUTPUT_DIR + "/train_dataset.npz", X=X_train_scaled, y=y_train)
np.savez(OUTPUT_DIR + "/val_dataset.npz", X=X_val_scaled, y=y_val)
np.savez(OUTPUT_DIR + "/test_dataset.npz", X=X_test, y=y_test)
print("Datasets saved.")

################## Training Set ##################
No file found for sensor 'Head' in run '0721_0853_noheadsensor'. Returning empty DataFrame.
Skipping run 0721_0853_noheadsensor due to missing sensor data.
-------------------------------------------
Label file exists for 0724_0723_sequences.json
Synch file and selected peaks file exists for 0724_0723
Start value video/label vector: 2184, Start value sensor: 1340
Created 5724 samples from run 0724_0723
Shape of X: (5724, 50, 27), Shape of y: (5724,)
-------------------------------------------
Label file exists for 0720_0828_sequences.json
Synch file and selected peaks file exists for 0720_0828
Start value video/label vector: 1886, Start value sensor: 1201
Created 5799 samples from run 0720_0828
Shape of X: (5799, 50, 27), Shape of y: (5799,)
-------------------------------------------
Label file exists for 0720_0858_sequences.json
Synch file and selected peaks file exists for 0720_0858
KeyError: 'Head' in synch or selected peaks file f

C:\Users\zane2\AppData\Local\Temp\ipykernel_21136\1219513351.py:51: DtypeWarning: Columns (5,6,7,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(matches[0], skiprows=header_line, usecols=cols)
C:\Users\zane2\AppData\Local\Temp\ipykernel_21136\1219513351.py:51: DtypeWarning: Columns (5,6,7,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(matches[0], skiprows=header_line, usecols=cols)


Label file exists for 0727_0735_sequences.json
Synch file and selected peaks file exists for 0727_0735
Start value video/label vector: 1706, Start value sensor: 1100
Created 6401 samples from run 0727_0735
Shape of X: (6401, 50, 27), Shape of y: (6401,)
-------------------------------------------
Label file exists for 0713_0833_sequences.json
Synch file and selected peaks file exists for 0713_0833
Start value video/label vector: 3326, Start value sensor: 2562
Created 5670 samples from run 0713_0833
Shape of X: (5670, 50, 27), Shape of y: (5670,)
-------------------------------------------
Label file exists for 0717_0717_sequences.json
Synch file and selected peaks file exists for 0717_0717
Start value video/label vector: 2544, Start value sensor: 1344
Created 5342 samples from run 0717_0717
Shape of X: (5342, 50, 27), Shape of y: (5342,)
-------------------------------------------
3 runs out of 3 processed for test set.


Saving datasets to datasets_output...
Datasets saved.
